# Perbandingan metode aktivitas manusia dengan YOLO

Jalankan menggunakan **HumanActivity Research GPU**. Default E03 adalah YOLO11n deteksi orang -> crop -> MobileNetV2 frozen -> GRU yang dilatih sendiri. E01 dan E02 tersedia sebagai pembanding pada manifest yang sama. Enam kelas KTH; idle dan data pribadi tidak dimasukkan agar perbandingan utama konsisten.

Model YOLO pralatih mendeteksi orang, bukan langsung mengenali enam aktivitas. Dataset dan checkpoint sudah tersedia lokal. Hasil lama tidak ditimpa.

1. Jalankan sel berurutan; periksa pilot dan manifest.
2. Pilih E01, E02, atau E03 pada sel konfigurasi.
3. Jalankan ekstraksi fitur dan training; fitur disimpan agar tidak dihitung ulang.
4. Konfigurasi protokol awal sudah dibekukan. Test dijalankan setelah pemilihan checkpoint dari validasi. Jangan gunakan hasil test untuk tuning.
5. Untuk metode lain ubah EXPERIMENT, lalu jalankan ulang mulai sel fitur. Jangan jalankan dua training sekaligus.


In [ ]:
import os
os.environ['KERAS_BACKEND']='torch'
from pathlib import Path
import sys,json
HERE=Path.cwd().resolve()
if not (HERE/'experiments.py').exists():
    raise RuntimeError('Jalankan Jupyter dari notebooks/research/.')
sys.path.insert(0,str(HERE))
import experiments as ex
EXPERIMENT='E03'  # E01 ANN, E02 full-frame GRU, E03 YOLO crop GRU
SEED=42
EPOCHS=40
FINAL_TEST=True  # protokol awal sudah dibekukan; False jika masih melakukan tuning
assert EXPERIMENT in ex.METHODS
print(ex.check_gpu())
print('Eksperimen:',EXPERIMENT)


## Pilot deteksi dan pose
Coverage pilot bukan akurasi aktivitas. Periksa bahwa kotak mengelilingi orang yang benar. Sampel pilot hanya train/val.


In [ ]:
from IPython.display import display,Image
import pandas as pd
if not (HERE/'pilot/pilot.csv').exists():ex.pilot()
pilot=pd.read_csv(HERE/'pilot/pilot.csv')
display(pilot)
display(Image(filename=str(HERE/'pilot/detector_contact_sheet.jpg'),width=1000))
print((HERE/'pilot/summary.json').read_text())


## Manifest jendela bersama
Video dibaca berurutan untuk memverifikasi prefix yang dapat didekode. Bagian akhir yang gagal didekode tidak dipakai oleh seluruh metode. Manifest ber-hash disimpan dan digunakan ulang.


In [ ]:
manifest=ex.prepare_protocol()
display(pd.crosstab(manifest.action,manifest.split))
print('Jendela:',len(manifest))
print((HERE/'protocol/protocol.json').read_text())


## Ekstraksi fitur
E01 dan E02 memakai cache full frame yang sama. E03 memakai cache crop terpisah. ROI tidak terdeteksi diisi frame asli untuk bentuk tensor, tetapi coverage dicatat: training hanya memakai coverage >=80%, evaluasi utama menghitung coverage kurang dari itu sebagai kegagalan. Tidak ada test sulit yang dibuang diam-diam.


In [ ]:
representation,kind=ex.METHODS[EXPERIMENT]
X,coverage,feature_config=ex.extract_features(manifest,representation)
print('Fitur:',X.shape)
quality=manifest[['split','action']].copy()
quality['coverage']=coverage
display(quality.groupby(['split','action']).coverage.agg(['mean','min']))


## Training head sendiri
Early stopping dan checkpoint memakai validation macro F1, validation loss sebagai tie-breaker. CNN dan YOLO dibekukan. E02 dan E03 memakai head GRU yang identik.


In [ ]:
model,run_dir=ex.train_experiment(EXPERIMENT,manifest,X,coverage,feature_config,seed=SEED,epochs=EPOCHS)
history=pd.read_json(run_dir/'history.json')
display(history)
history.plot(x='epoch',y=['loss','val_loss'],title=EXPERIMENT+' loss')
history.plot(x='epoch',y='val_macro_f1',title=EXPERIMENT+' validation macro F1')
print('Checkpoint:',run_dir/'best.keras')


## Evaluasi test setelah konfigurasi dibekukan
Test KTH sudah pernah dilihat pada pengembangan sebelumnya. Hasil ini adalah benchmark berulang, bukan blind test kamera pribadi. Jangan membandingkan langsung angka ini dengan baseline lama yang menggunakan unit subsekuens berbeda.


In [ ]:
if FINAL_TEST:
    metrics=ex.evaluate(model,run_dir,manifest,X,coverage)
    print(json.dumps(metrics,indent=2))
    display(pd.read_csv(run_dir/'confusion_matrix.csv',index_col=0))
else:
    print('Test belum dijalankan. Bekukan konfigurasi, set FINAL_TEST=True, lalu jalankan sel ini.')


## Ringkasan seluruh eksperimen yang sudah dievaluasi
Tabel hanya memuat run yang benar-benar memiliki hasil test. Waktu training bukan latensi inferensi. Pengujian video dan profiler akan memakai pipeline lengkap, termasuk detektor, CNN dan head.


In [ ]:
rows=[]
for file in sorted((HERE/'results').glob('E*/seed_*/test_metrics.json')):
    metrics=json.loads(file.read_text())
    cfg=json.loads((file.parent/'run_config.json').read_text())
    rows.append({'experiment':cfg['experiment'],'seed':cfg['seed'],'accuracy':metrics['accuracy'],'macro_f1':metrics['macro_f1'],'balanced_accuracy':metrics['balanced_accuracy'],'coverage':metrics['coverage'],'training_seconds':cfg['training_seconds']})
display(pd.DataFrame(rows))


## Video uji setelah training
Menggunakan video yang sama di folder inference proyek sebelumnya. Ini demonstrasi, bukan metrik akurasi tanpa anotasi independen. E03 menolak jendela jika deteksi orang kurang dari 80%. Model utama enam kelas belum mengenali idle.


In [ ]:
VIDEO_UJI=HERE.parent/'inference'/'video_uji.mp4'
RUN_VIDEO=True
if RUN_VIDEO and VIDEO_UJI.exists():
    video_output=ex.infer_video(EXPERIMENT,VIDEO_UJI,seed=SEED)
    from IPython.display import Video
    display(Video(str(video_output),embed=False))
else:
    print('Untuk demonstrasi, set RUN_VIDEO=True dan jalankan sel ini setelah training.')


## Sumber dan batasan
Lihat SUMBER_DAN_REFERENSI.md, asset_manifest.json dan RENCANA_EKSEKUSI.md di folder ini. YOLO11 dan pose pralatih berasal dari Ultralytics. Pengujian awal single-person; belum ada pelacakan identitas banyak orang atau evaluasi kamera live. E04 pose masih menunggu analisis kualitas; notebook ini melaksanakan E01–E03.
